In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from src.config.settings import (
    LANDING_PATH,
    BRONZE_PATH,
    SILVER_PATH,
    QUARANTINE_PATH,
    QUALITY_REPORT_PATH,
    DATABASE_NAME,
    BRONZE_TABLE,
    SILVER_TABLE,
    QUARANTINE_TABLE
)

from src.quality.quality_rules import (
    validate_required_columns,
    standardize_taxi_data,
    add_quality_validation_columns,
    get_valid_records,
    get_invalid_records
)


In [0]:
%run ./ingestion_to_land

In [0]:
spark = SparkSession.builder.appName("ifood-case-taxi").getOrCreate()

In [0]:

def create_database() -> None:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")

def ingest_bronze() -> None:
    """
    Lê os arquivos originais da Landing Zone e grava a camada Bronze.
    """

    df_raw = (
        spark.read
        .option("mergeSchema", "true")
        .format("parquet")
        .load(LANDING_PATH)
    )

    validate_required_columns(df_raw)

    df_bronze = (
        df_raw
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.input_file_name())
    )

    (
        df_bronze.write
        .mode("overwrite")
        .format("delta")
        .save(BRONZE_PATH)
    )

    print("Camada Bronze criada com sucesso.")
def build_silver_and_quarantine() -> None:
    """
    Lê a Bronze, aplica padronização, validações de qualidade
    e separa registros válidos e inválidos.
    """
    df_bronze = spark.read.format("delta").load(BRONZE_PATH)

    df_standardized = standardize_taxi_data(df_bronze)

    df_validated = add_quality_validation_columns(df_standardized)

    df_valid = get_valid_records(df_validated)
    df_invalid = get_invalid_records(df_validated)

    (
        df_valid.write
        .mode("overwrite")
        .format("delta")
        .partitionBy("pickup_month")
        .save(SILVER_PATH)
    )

    (
        df_invalid.write
        .mode("overwrite")
        .format("delta")
        .partitionBy("pickup_month")
        .save(QUARANTINE_PATH)
    )

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
        USING DELTA
        LOCATION '{SILVER_PATH}'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE}
        USING DELTA
        LOCATION '{QUARANTINE_PATH}'
    """)

    print("Camada Silver criada com sucesso.")
    print("Camada Quarantine criada com sucesso.")


def generate_quality_report() -> None:
    """
    Gera relatório simples de qualidade da carga.
    """
    df_valid = spark.read.format("delta").load(SILVER_PATH)
    df_invalid = spark.read.format("delta").load(QUARANTINE_PATH)

    valid_count = df_valid.count()
    invalid_count = df_invalid.count()
    total_count = valid_count + invalid_count

    valid_percentage = round((valid_count / total_count) * 100, 2) if total_count > 0 else 0
    invalid_percentage = round((invalid_count / total_count) * 100, 2) if total_count > 0 else 0

    summary_df = spark.createDataFrame(
        [
            (
                total_count,
                valid_count,
                invalid_count,
                valid_percentage,
                invalid_percentage
            )
        ],
        [
            "total_records",
            "valid_records",
            "invalid_records",
            "valid_percentage",
            "invalid_percentage"
        ]
    ).withColumn("report_timestamp", F.current_timestamp())

    rejection_df = (
        df_invalid
        .groupBy("pickup_month", "rejection_reason")
        .agg(F.count("*").alias("rejected_records"))
        .orderBy("pickup_month", F.desc("rejected_records"))
    )

    (
        summary_df.write
        .mode("overwrite")
        .format("delta")
        .save(f"{QUALITY_REPORT_PATH}/summary")
    )

    (
        rejection_df.write
        .mode("overwrite")
        .format("delta")
        .save(f"{QUALITY_REPORT_PATH}/rejections")
    )

    print("Resumo de qualidade:")
    summary_df.show(truncate=False)

    print("Rejeições por motivo:")
    rejection_df.show(truncate=False)


def run_pipeline() -> None:
    create_database()

    print("Etapa 1 - Download dos arquivos para Landing Zone")
    ingestion_to_landing()

    print("Etapa 2 - Ingestão Bronze")
    ingest_bronze()

    print("Etapa 3 - Construção Silver e Quarantine")
    build_silver_and_quarantine()

    print("Etapa 4 - Relatório de qualidade")
    generate_quality_report()

    print("Pipeline executado com sucesso.")


In [0]:
run_pipeline()